In [ ]:
from pathlib import Path
from typing import Union, Optional
import pandas as pd
import numpy as np
import logging
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [ ]:
# Use pathlib for robust path handling
BASE_DIR = Path.cwd().parent 
PROCESSED_DATA_DIR = BASE_DIR / "data_processed"
CONFIG_DIR = BASE_DIR / "config"
BRFSS_ZIP_FILE = PROCESSED_DATA_DIR / "BRFSS_2015_2024.zip"
BRFSS_CLEAN_ZIP_FILE = PROCESSED_DATA_DIR / "BRFSS_2015_2024_cleaned.zip"
RECODE_MAP_FILE = CONFIG_DIR / "recode_mappings.json"

# Ensure directories exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def read_zipped_csv(
    zip_path: Union[str, Path],
    **read_csv_kwargs
) -> pd.DataFrame:
    """
    Read a single-CSV .zip file into a pandas DataFrame.

    Parameters
    ----------
    zip_path : str or Path
        Path to the .zip file (containing exactly one CSV, or a CSV
        with the same name as the zip).
    **read_csv_kwargs :
        Any extra keyword args passed through to pandas.read_csv
        (e.g. sep, dtype, parse_dates).

    Returns
    -------
    pd.DataFrame
    """
    zip_path = Path(zip_path)
    # For a single CSV in the zip, this is enough:
    return pd.read_csv(zip_path, compression="zip", **read_csv_kwargs)

In [ ]:
def write_df_to_zipped_csv(
    df: pd.DataFrame,
    zip_path: Union[str, Path],
    csv_name: str | None = None,
    **to_csv_kwargs,
) -> None:
    """
    Write a DataFrame to a CSV stored inside a .zip file.

    Parameters
    ----------
    df : pd.DataFrame
        Data to write.
    zip_path : str or Path
        Path to the .zip file to create, e.g. 'data.zip'.
    csv_name : str, optional
        Name of the CSV file inside the zip, e.g. 'data.csv'.
        If None, uses the zip file stem with '.csv'.
    **to_csv_kwargs :
        Extra keyword args passed to DataFrame.to_csv
        (e.g. index=False, sep=',', encoding='utf-8').
    """
    zip_path = Path(zip_path)
    if csv_name is None:
        csv_name = zip_path.stem + ".csv"

    compression_opts = {
        "method": "zip",
        "archive_name": csv_name,
    }

    df.to_csv(
        zip_path,
        header=True,
        compression=compression_opts,
        **to_csv_kwargs,
    )

In [ ]:
# Load the BRFSS data
df = read_zipped_csv(BRFSS_ZIP_FILE)
logging.info(f"Initial data loaded with {df.shape[0]} rows. and columns: {df.shape[1]}")

In [ ]:
def print_unique_values(data, columnName):
    unique_column_values = data[columnName].unique()
    if pd.api.types.is_numeric_dtype(unique_column_values):
        unique_column_values = np.sort(unique_column_values)
    column_datatype = data[columnName].dtype
    num_unique_column_values = len(unique_column_values)
    print(f'{num_unique_column_values} Unique values in column {columnName} of type {column_datatype}: {unique_column_values}')

# For each column, print unique values
for col in df.columns:
    print_unique_values(df, col)

In [ ]:
# For each column, replace 'Not Sure' or 'Refused' codes with NaN

# replace_unsure_refused = {
#     'HEALTH_STATUS': {9: np.nan},
#     'PHYSICAL_HEALTH_STATUS': {9: np.nan},
#     'MENTAL_HEALTH_STATUS': {9: np.nan},
#     'EXERCISE': {9: np.nan},
#     'HEALTH_CARE_COVERAGE': {9: np.nan},
#     'SMOKER': {9: np.nan},
#     'DRINKER': {7: np.nan, 9: np.nan},
#     'SOCIAL_DRINKER': {9: np.nan},
#     'HEAVY_ALCOHOL_CONSUMPTION': {9: np.nan},
#     #'HEART_ATTACK': {},
#     'STROKE': {7: np.nan, 9: np.nan},
#     'DIABETES': {7: np.nan, 9: np.nan},
#     #'ARTHRITIS': {},
#     'MARITAL_STATUS': {9: np.nan},
#     'EMPLOYMENT': {9: np.nan},
#     'SEX': {7: np.nan, 9: np.nan},
#     'AGE_CATEGORIES': {14: np.nan},
#     #'BMICAT': {},
#     'EDUCATION_LEVEL': {9: np.nan},
#     'INCOME': {9: np.nan}
# }

# df.replace(replace_unsure_refused, inplace=True)

In [ ]:
def recode_column(df, col_name, old_values, new_values):
    if len(old_values) != len(new_values):
        raise ValueError("old_values and new_values must have same length")
    mapping = dict(zip(old_values, new_values))
    df[col_name] = df[col_name].replace(mapping)
    return df

def apply_recode_map(df, map_file=RECODE_MAP_FILE):
    with open(map_file, 'r') as f:
        recode_map = json.load(f)
    
    for col_name, mapping in recode_map.items():
        if col_name not in df.columns:
            continue

        # is_numeric = pd.api.types.is_numeric_dtype(df[col_name])

        if mapping.get('multi_step'):
            # Handle grouped (e.g. SMOKER)
            no_codes = mapping['no_codes']
            yes_codes = mapping['yes_codes']

            # if is_numeric:
            #     no_codes = [float(x) for x in no_codes]
            #     yes_codes = [float(x) for x in yes_codes]

            df.loc[df[col_name].isin(no_codes), col_name] = 0
            df.loc[df[col_name].isin(yes_codes), col_name] = 1
        else:
            # Direct 1:1
            old_vals = mapping['old_values']
            new_vals = mapping['new_values']
            df = recode_column(df, col_name, old_vals, new_vals)
    
        missing_vals = mapping.get('missing_values', [])
        if missing_vals:
            # if is_numeric:
            #     # Convert config strings ("77", "99") to floats (77.0, 99.0)
            #     missing_vals = [float(x) for x in missing_vals]
            
            # Replace only in this specific column
            df[col_name] = df[col_name].replace(missing_vals, np.nan)
    
    return df

# Apply to your DataFrame
df = apply_recode_map(df)

In [ ]:
# For each column, print unique values
for col in df.columns:
    print_unique_values(df, col)

In [ ]:
# Drop duplicate rows
logging.info("Dropping duplicate rows from the dataset.")
initial_row_count = df.shape[0]
df.drop_duplicates(inplace=True)
logging.info(f"Data after dropping duplicates has {df.shape[0]} rows (dropped {initial_row_count - df.shape[0]} duplicates).")

In [ ]:
#Impute missing values using SMOTE
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
# Separate features and target variable
X = df.drop('DIABETES', axis=1)
y = df['DIABETES']
X_resampled, y_resampled = smote.fit_resample(X, y)
# Combine resampled features and target back into a single DataFrame
df = pd.concat([X_resampled, y_resampled], axis=1)
logging.info(f"Data after SMOTE resampling has {df.shape[0]} rows.")
# Save cleaned data to zipped CSV
write_df_to_zipped_csv(df, BRFSS_CLEAN_ZIP_FILE, index=False)
logging.info(f"Cleaned data saved to {BRFSS_CLEAN_ZIP_FILE}.")
logging.info("Data description:\n")
df.describe().T

In [ ]:
# Cleanse and process the data as needed
logging.info("Dropping rows with missing data.")
initial_row_count = df.shape[0]
df.dropna(inplace=True)
logging.info(f"Data after dropping missing rows has {df.shape[0]} rows (dropped {initial_row_count - df.shape[0]} missing data rows).")

In [ ]:
# For each column, print unique values
for col in df.columns:
    print_unique_values(df, col)

In [ ]:
logging.info("Data information:\n")
df.info()

In [ ]:
logging.info("Data description:\n")
df.describe().T

In [ ]:
logging.info("First few rows of the dataset:")
df.head()

In [ ]:
logging.info(f"The BRFSS 2015-2024 dataset has been successfully prepared and is ready for analysis and had {df.shape[0]} rows and {df.shape[1]} columns.")

In [ ]:
logging.info("Write the cleaned dataset to csv")
write_df_to_zipped_csv(df, BRFSS_CLEAN_ZIP_FILE, index=False)

In [ ]:
plot_data = pd.crosstab(df['YEAR'], df['DIABETES'])
sns.set_style("whitegrid")
ax = plot_data.plot(kind='bar', stacked=True, figsize=(12, 7), width=0.75)
plt.title('Diabetes Distribution Over Years', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Total Records', fontsize=12)
plt.xticks(rotation=0)
plt.legend(title='Diabetes Status', labels=['Non-Diabetes', 'Diabetes'], 
               bbox_to_anchor=(1.02, 1), loc='upper left')
totals = plot_data.sum(axis=1)
for container in ax.containers:
    labels = []
    for i, bar in enumerate(container):
        height = bar.get_height()
        year_idx = plot_data.index[i]
        total = totals[year_idx]
        
        # Calculate percentage
        pct = 0
        if total > 0:
            pct = (height / total) * 100
        
        # Create label text
        # We only show the label if the segment is large enough (>4%) to be readable
        if pct > 4: 
            labels.append(f'{pct:.1f}%')
        else:
            labels.append('')
    
    # Apply labels to the center of each bar segment
    ax.bar_label(container, labels=labels, label_type='center', 
                    fontsize=10, fontweight='bold', color='black')

plt.tight_layout()
plt.show()

In [ ]:
def sample_and_decode(df: pd.DataFrame,
                      year_col: str = "YEAR",
                      mapping_path: str | Path = "VALUE_RECODED_TEXT_MAP.json",
                      n_per_year: int = 2,
                      random_state: int | None = None) -> pd.DataFrame:
    """
    For each distinct value in `year_col`, sample `n_per_year` random rows,
    then replace enum codes using VALUE_RECODED_TEXT_MAP.json.
    """

    # 1) Load mapping file (expects structure like: { "DIABETES": {"0": "No", "1": "Yes"}, ... })
    mapping_path = Path(mapping_path)
    with mapping_path.open("r") as f:
        value_map = json.load(f)   # [file:2]

    # 2) Sample n_per_year rows for each year
    # If a year has fewer than n_per_year rows, it takes all available rows
    sampled = (
        df
        .groupby(year_col, group_keys=False)
        .apply(lambda g: g.sample(
            n=min(len(g), n_per_year),
            random_state=random_state
        ))
        .reset_index(drop=True)
    )  # [file:1]

    # 3) Apply enum→text mapping column by column
    for col, mapping in value_map.items():
        if col not in sampled.columns:
            continue

        # normalize keys of mapping to strings, then map from df values cast to int/str
        # mapping example in VALUE_RECODED_TEXT_MAP.json: "DIABETES 0 No, 1 Yes, ..." [file:2]
        # So we build {0: "No", 1: "Yes"} etc.
        col_map = {}
        for k, v in mapping.items():
            try:
                # try numeric key
                col_map[int(k)] = v
            except (ValueError, TypeError):
                # fall back to string key
                col_map[str(k)] = v

        # if column is numeric (e.g. float codes 0.0, 1.0) coerce to int where possible
        if pd.api.types.is_numeric_dtype(sampled[col]):
            sampled[col] = (
                sampled[col]
                .dropna()
                .astype(int)
                .map(col_map)
                .reindex(sampled.index)
            )
        else:
            sampled[col] = sampled[col].map(col_map)

    return sampled

decoded_sample = sample_and_decode(
    df,
    year_col="YEAR",
    mapping_path=CONFIG_DIR / "VALUE_RECODED_TEXT_MAP.json",
    n_per_year=2,
    random_state=42,
)
decoded_sample.head(n=20)

In [ ]:
for col in decoded_sample.columns:
    print_unique_values(decoded_sample, col)